<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/12-caching-memory/03-checkpointing-vs-caching.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 12.3 — Checkpointing vs caching: sever the lineage

Grow a plan for 25 iterations and watch `explain()` output balloon,
cache it (lineage still there), then checkpoint it and watch the plan
collapse back to two lines. Closes with the nondeterminism
correctness bug caching can't fix but checkpointing does.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("12.3-checkpointing-vs-caching")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
# local[*] note: one JVM plays driver + all executors, so the Executors tab
# shows a single "driver" entry — the memory MODEL below is identical to a
# real cluster, only the topology is collapsed.

In [ ]:
import shutil
shutil.rmtree("checkpoints", ignore_errors=True)
spark.sparkContext.setCheckpointDir("checkpoints")   # local dir for teaching only —
                                                       # production uses HDFS/S3 (durable storage)

## Grow a long lineage: 25 iterations, one DataFrame

Each loop adds a layer to the logical plan. Nothing is cached or
checkpointed yet — watch the plan string length as a proxy for
planning cost.

In [ ]:
df = spark.range(200_000).withColumn("v", col("id") * 2)

for i in range(25):
    df = df.withColumn(f"v{i}", col("v") + i)   # one more layer of plan every iteration

plan_len = len(df.queryExecution.optimizedPlan.toString())
print(f"optimized plan length after 25 iterations: {plan_len:,} characters")
print("last 300 chars of the plan (truncated for readability):")
print(df.queryExecution.optimizedPlan.toString()[-300:])

## Caching doesn't shrink the plan — only the recompute cost

Cache it, force materialization, then keep extending it 10 more
iterations. The plan keeps growing from wherever caching stopped.

In [ ]:
df.cache()
df.count()   # materialize the first 25 layers

for i in range(25, 35):
    df = df.withColumn(f"v{i}", col("v") + i)

plan_len2 = len(df.queryExecution.optimizedPlan.toString())
print(f"plan length after caching + 10 more iterations: {plan_len2:,} characters")
print("still growing -- cache changed WHAT gets recomputed, not the plan's size")

## Checkpoint: the plan collapses to "read the checkpoint"

`checkpoint(eager=True)` writes current data to `checkpoints/` and
replaces the plan. Compare plan length before and after.

In [ ]:
before = len(df.queryExecution.optimizedPlan.toString())

df = df.checkpoint(eager=True)   # blocks until written; truncates lineage

after = len(df.queryExecution.optimizedPlan.toString())
print(f"plan length before checkpoint: {before:,} characters")
print(f"plan length after  checkpoint: {after:,} characters")
print("\nfull post-checkpoint plan (should be short — a scan of the checkpoint files):")
print(df.queryExecution.optimizedPlan.toString())

## Extend again after checkpointing — the new plan starts fresh

In [ ]:
for i in range(35, 40):
    df = df.withColumn(f"v{i}", col("v") + i)

print("plan length, 5 more iterations on TOP of the checkpoint:",
      len(df.queryExecution.optimizedPlan.toString()), "characters")
print("-- compare to the pre-checkpoint growth rate above")

## `localCheckpoint()` — truncation without the durability guarantee

Same lineage-severing effect, written to executor-local storage
instead. Good for dev/iteration; does not survive executor loss.

In [ ]:
df2 = spark.range(100_000)
for i in range(15):
    df2 = df2.withColumn(f"w{i}", col("id") + i)

before2 = len(df2.queryExecution.optimizedPlan.toString())
df2 = df2.localCheckpoint()
after2 = len(df2.queryExecution.optimizedPlan.toString())
print(f"localCheckpoint: {before2:,} chars -> {after2:,} chars")

## The correctness bug caching can't fix, checkpointing can

A lineage with nondeterminism: cache it, force an eviction-triggering
memory squeeze, and watch a later action potentially recompute
different values. Checkpointing freezes the result instead.

In [ ]:
nondeterm = spark.range(500_000).withColumn("r", F.rand(seed=None))  # no fixed seed on purpose
nondeterm.cache()
first_sample = nondeterm.orderBy("id").limit(3).collect()
print("first read:", [round(r["r"], 6) for r in first_sample])

nondeterm.unpersist()   # simulate eviction: force a "recompute" on next access
second_sample = nondeterm.orderBy("id").limit(3).collect()
print("after unpersist + reaccess (recomputed):",
      [round(r["r"], 6) for r in second_sample])
print("-> values differ: cache does NOT freeze nondeterministic lineage (2.4's warning)")

nondeterm_ckpt = nondeterm.checkpoint(eager=True)   # materializes ONCE, permanently
a = nondeterm_ckpt.orderBy("id").limit(3).collect()
b = nondeterm_ckpt.orderBy("id").limit(3).collect()
print("\ncheckpointed, read twice:",
      [round(r["r"], 6) for r in a], "==", [round(r["r"], 6) for r in b])

In [ ]:
spark.stop()
shutil.rmtree("checkpoints", ignore_errors=True)   # self-clean

## Exercises

1. Time `df.queryExecution.optimizedPlan` access (not `explain()`,
   just building the string) at iteration 5, 15, 25, 35 of the growth
   loop. Does the cost grow linearly or worse?
2. Repeat the "cache doesn't shrink the plan" demo, but this time
   simulate an executor loss by `unpersist()`-ing mid-lineage and
   re-running the final action. Time it against the checkpointed
   equivalent.
3. Build a lazy checkpoint (`eager=False`) inside a loop that keeps
   extending the DataFrame for 3 more iterations before the next
   action. Show that more lineage accumulated than a caller might
   expect before the checkpoint actually fires.
4. Cache THEN checkpoint the same long-lineage DataFrame (cache first,
   checkpoint second, per the lesson's ordering advice). Time it
   against checkpointing a cold (uncached) equivalent — confirm the
   cache-first order is cheaper.
5. Reproduce the nondeterminism bug with `F.monotonically_increasing_id()`
   instead of `F.rand()` — does eviction-then-recompute change those
   values too? Why or why not (what's actually nondeterministic about
   each)?

In [ ]:
# your exercise code here